In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import tensorflow.keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

pi = 3.14159265359

maxval=1e9
minval=1e-9

2025-11-18 17:02:34.166817: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-18 17:02:35.146722: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
from dataloaders.OptimizedDataGenerator_v2p5 import OptimizedDataGenerator
from loss import *
from models.mlp_encoder_model_nonquantized import *

In [3]:
seed = 10
tf.random.set_seed(seed)
random.seed(seed)

In [4]:
dataset_base_dir = "/data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train_contained_digitize-manual_mlp-SLIM")
dataset_validation_dir = os.path.join(dataset_base_dir, "test_contained_digitize-manual_mlp-SLIM")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train_contained_digitize-manual_mlp-SLIM")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val_contained_digitize-manual_mlp-SLIM")

dirs_to_create = [
    tfrecords_dir_train,
    tfrecords_dir_val,
    dataset_train_dir,
    dataset_validation_dir
]

# Create each directory if it doesn't exist
for directory in dirs_to_create:
    os.makedirs(directory, exist_ok=True)

In [5]:
print(f'Number of training files: {len(os.listdir(dataset_train_dir))}')
print(f'Number of validation files: {len(os.listdir(dataset_validation_dir))}')

Number of training files: 80
Number of validation files: 20


In [6]:
batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_validation_dir))

In [7]:
start_time = time.time()
validation_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_validation_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = val_batch_size,
    file_count = val_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshod = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, 
    files_from_end = True,

    tfrecords_dir = tfrecords_dir_val,
    use_time_stamps = [0,19],
    max_workers = 1,
    #load_from_tfrecords_dir = tfrecords_dir_val
)

print("--- Validation generator %s seconds ---" % (time.time() - start_time))

Processing Files...: 100%|██████████| 20/20 [00:09<00:00,  2.20it/s]


Directory /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_digitize-manual_mlp-SLIM is removed...


Saving batches as TFRecords: 100%|██████████| 24/24 [00:06<00:00,  3.75it/s]


Metadata saved successfully ast /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_digitize-manual_mlp-SLIM/metadata.json
Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_digitize-manual_mlp-SLIM/metadata.json
--- Validation generator 15.763993263244629 seconds ---


In [8]:
# training generator
start_time = time.time()
training_generator = OptimizedDataGenerator(
    dataset_base_dir = dataset_train_dir,
    file_type = "parquet",
    data_format = "3D",
    batch_size = batch_size,
    file_count = train_file_size,
    to_standardize = False, # False when processing manually digitized inputs
    log_compression = False, # False when processing manually digitized inputs
    select_contained = True,
    noise = -1,
    min_threshold = None,
    max_threshold = None,
    include_y_local= False,
    labels_list = ['x-midplane','y-midplane','cotBeta'],
    input_shape = (2,16,16), # (20,16,16),
    transpose = (0,2,3,1),
    shuffle = False, # True 

    tfrecords_dir = tfrecords_dir_train,
    use_time_stamps = [0,19],
    max_workers = 1,
    #load_from_tfrecords_dir = tfrecords_dir_train
)
print("--- Training generator %s seconds ---" % (time.time() - start_time))

Processing Files...: 100%|██████████| 80/80 [00:40<00:00,  1.96it/s]


Directory /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_digitize-manual_mlp-SLIM is removed...


Saving batches as TFRecords: 100%|██████████| 95/95 [00:26<00:00,  3.61it/s]


Metadata saved successfully ast /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_digitize-manual_mlp-SLIM/metadata.json
Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_digitize-manual_mlp-SLIM/metadata.json
--- Training generator 67.49560618400574 seconds ---


In [9]:
training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle = True,
    seed = seed,
    quantize = False
)

validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_val,
    shuffle = True,
    seed = seed,
    quantize = False
)


Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_train_contained_digitize-manual_mlp-SLIM/metadata.json


Loading metadata from /data/dajiang/smart-pixels/largerWindowPreliminary/dataset_3sr_16x16_50x12P5_centeredIncidence_-8_8_offsets_parquets/TFR_files/2t/TFR_val_contained_digitize-manual_mlp-SLIM/metadata.json


In [10]:
model=CreateModel_Slim((16,16,2))
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_sse_loss
)

model.summary()

Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls (InputLayer)     [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['input_pxls[0][0]']          
 Pooling2D)                                                                                       
                                                                                                  
 average_pooling2d_1 (Avera  (None, 1, 16, 2)             0         ['input_pxls[0][0]']          
 gePooling2D)                                                                                     
                                                                                 

In [11]:
# training
pitch = '50x12P5'
fingerprint = '%08x' % random.randrange(16**8)
base_dir = '/data/dajiang/smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_-8_8_offsets_weights/'
weights_dir = base_dir + 'weights-{}-bs{}-{}-2t-mlp_SLIM-manual_input_digitization-checkpoints'.format(pitch, batch_size, fingerprint)

# create output directories
if os.path.isdir(base_dir):
    os.mkdir(weights_dir)
else:
    os.mkdir(base_dir)
    os.mkdir(weights_dir)
    
checkpoint_filepath = weights_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'
mcp = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=False,
)

print('Model fingerprint: {}'.format(fingerprint))

Model fingerprint: 34c2da80


In [ ]:
history = model.fit(x=training_generator,
                    validation_data=validation_generator,
                    callbacks=[mcp],
                    epochs=1000,
                    shuffle=False, # shuffling now occurs within the data-loader
                    verbose=1)

Epoch 1/1000
95/95 [==============================] - 15s 141ms/step - loss: 725.4428 - val_loss: 523.1516
Epoch 2/1000
95/95 [==============================] - 13s 133ms/step - loss: 477.6101 - val_loss: 430.7447
Epoch 3/1000
95/95 [==============================] - 12s 124ms/step - loss: 377.4533 - val_loss: 322.4537
Epoch 4/1000
95/95 [==============================] - 13s 132ms/step - loss: 283.8800 - val_loss: 255.1520
Epoch 5/1000
95/95 [==============================] - 12s 124ms/step - loss: 241.7943 - val_loss: 230.4010
Epoch 6/1000
95/95 [==============================] - 12s 127ms/step - loss: 223.3942 - val_loss: 216.8163
Epoch 7/1000
95/95 [==============================] - 11s 115ms/step - loss: 212.6797 - val_loss: 208.5427
Epoch 8/1000
95/95 [==============================] - 12s 122ms/step - loss: 205.9416 - val_loss: 202.9151
Epoch 9/1000
95/95 [==============================] - 12s 122ms/step - loss: 201.1366 - val_loss: 198.6810
Epoch 10/1000
95/95 [================